<a href="https://colab.research.google.com/github/SteFabolous/stem-splitter-with-google-colab/blob/main/stem_splitter_with_google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⚙️ Step 1: Connect Google Drive, Install Dependencies & Setup Environment

import os
import sys
import subprocess
from google.colab import drive

print("⚙️ Initializing environment...")

# Mount Google Drive
drive.mount('/content/drive')

print("📦 Installing dependencies...")

# Install audio-separator and supporting tools
!pip install -q "audio-separator[gpu]" yt-dlp spotdl ml_collections \
    "nvidia-cublas-cu12" "nvidia-cudnn-cu12"

# ------------------------------------------------------------
# IMPORTANT:
# Pin ONNX Runtime to 1.26.x.
#
# 1.26.x = CUDA 12.x + cuDNN 9.x
# 1.27+ = CUDA 13.x on PyPI
# ------------------------------------------------------------

!pip uninstall -y onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q --force-reinstall "onnxruntime-gpu==1.26.0"

# ------------------------------------------------------------
# Configure NVIDIA library paths
# ------------------------------------------------------------

py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
site_packages = f"/usr/local/lib/python{py_ver}/dist-packages"
nvidia_dir = os.path.join(site_packages, "nvidia")

lib_paths = []

if os.path.exists(nvidia_dir):
    for pkg in os.listdir(nvidia_dir):
        pkg_lib = os.path.join(nvidia_dir, pkg, "lib")
        if os.path.isdir(pkg_lib):
            lib_paths.append(pkg_lib)

# System CUDA libraries
if os.path.exists("/usr/local/cuda/lib64"):
    lib_paths.append("/usr/local/cuda/lib64")

# Remove duplicates while preserving order
lib_paths = list(dict.fromkeys(lib_paths))

if lib_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(lib_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# ------------------------------------------------------------
# Workspace folders
# ------------------------------------------------------------

input_folder = "/content/drive/MyDrive/Input_Audio"
output_folder = "/content/drive/MyDrive/Stems_Output"

os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# ------------------------------------------------------------
# Environment verification
# ------------------------------------------------------------

print("\n🔍 Checking GPU / CUDA environment...\n")

!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

import torch
import onnxruntime as ort

print(f"\n🔥 PyTorch version: {torch.__version__}")
print(f"🔥 PyTorch CUDA: {torch.version.cuda}")
print(f"🔥 ONNX Runtime: {ort.__version__}")
print(f"🔥 ONNX Runtime providers: {ort.get_available_providers()}")

if "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError(
        "❌ CUDAExecutionProvider is not available.\n"
        "The Colab environment could not initialize ONNX Runtime with CUDA."
    )

print("\n✅ CUDA / ONNX Runtime GPU setup successful!")
print("📁 Input folder :", input_folder)
print("📁 Output folder:", output_folder)
print("\n➡️ Now run Step 2.")

In [ ]:
# @title 🎛️ Step 2: Configure & Run Stem Splitter

import os
import glob
import shutil

from audio_separator.separator import Separator
import onnxruntime as ort


# ------------------------------------------------------------
# Verify ONNX Runtime / CUDA
# ------------------------------------------------------------

print("🔎 Checking ONNX Runtime GPU support...")

providers = ort.get_available_providers()
print(f"   ONNX Runtime version: {ort.__version__}")
print(f"   Available providers: {providers}")

if "CUDAExecutionProvider" not in providers:
    raise RuntimeError(
        "❌ CUDAExecutionProvider is not available.\n"
        "GPU acceleration could not be initialized. "
        "Please make sure Step 1 completed successfully."
    )

print("✅ CUDAExecutionProvider detected. GPU acceleration enabled.\n")


# ------------------------------------------------------------
# Audio Source
# ------------------------------------------------------------

# @markdown ### 📁 Audio Source (Choose one)

# @markdown Input folder path on Google Drive (containing audio files):
audio_input_folder = "/content/drive/MyDrive/Input_Audio"  # @param {type:"string"}

# @markdown Paste a YouTube URL:
youtube_url = ""  # @param {type:"string"}

# @markdown Or paste a Spotify Track/Album/Playlist URL:
spotify_url = ""  # @param {type:"string"}


# ------------------------------------------------------------
# AI Model Selection
# ------------------------------------------------------------

# @markdown ---
# @markdown ### 🤖 Select AI Model

model_choice = "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)"  # @param ["MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)", "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)", "MelBand-RoFormer BigBeta 6 (Balanced Vocals)", "MelBand-RoFormer BigBeta 5e (Max Vocal Fullness)", "MelBand-RoFormer BigBeta 4 (Classic)", "MelBand-RoFormer BigBeta 3", "BS-RoFormer Leap (unwa SOTA)", "Kim-unwa FT2 (Vocal Hybrid)", "BS-RoFormer ViperX (Vocals)", "HTDemucs v4 FT (4 Stems)", "HTDemucs v4 6-Stems (Guitar/Piano)"]


# ------------------------------------------------------------
# Advanced Model & Output Parameters
# ------------------------------------------------------------

# @markdown ---
# @markdown ### ⚙️ Advanced Model & Output Parameters

# @markdown **Overlap**: Higher values can improve transitions at segment boundaries.
# @markdown Recommended default: 8.
overlap = 40  # @param {type:"slider", min:1, max:40, step:1}

# @markdown **Output Format**: Audio format for saved stems.
output_format = "wav"  # @param ["wav", "flac", "mp3"]

# @markdown **Chunk Size (Segment Size)**: Processing window size. Lower values save GPU VRAM.
chunk_size_choice = "529200"  # @param ["112455", "352800", "485100", "529200", "661500"]
chunk_size = int(chunk_size_choice)

# @markdown **Use TTA (Test-Time Augmentation)**: Runs additional inference passes for potentially cleaner separation.
# @markdown This significantly increases processing time.
use_tta = True  # @param {type:"boolean"}

# @markdown **Extract Instrumental**: Keep enabled to output both Vocals & Instrumental stems.
# @markdown If unchecked, outputs Vocals only.
extract_instrumental = True  # @param {type:"boolean"}


# ------------------------------------------------------------
# Main Output Folder
# ------------------------------------------------------------

# @markdown ---
# @markdown ### 💾 Main Output Folder on Google Drive

output_folder = "/content/drive/MyDrive/Stems_Output"  # @param {type:"string"}

os.makedirs(output_folder, exist_ok=True)


# ------------------------------------------------------------
# Model Mapping
# ------------------------------------------------------------

models_map = {
    # --- MelBand-RoFormer BigBeta Series ---
    "MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)": "melband_roformer_big_beta7.ckpt",
    "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)": "melband_roformer_big_beta6x.ckpt",
    "MelBand-RoFormer BigBeta 6 (Balanced Vocals)": "melband_roformer_big_beta6.ckpt",
    "MelBand-RoFormer BigBeta 5e (Max Vocal Fullness)": "melband_roformer_big_beta5e.ckpt",
    "MelBand-RoFormer BigBeta 4 (Classic)": "melband_roformer_big_beta4.ckpt",
    "MelBand-RoFormer BigBeta 3": "melband_roformer_big_beta3.ckpt",

    # --- Other RoFormer Models ---
    "BS-RoFormer Leap (unwa SOTA)": "bs_roformer_leap.ckpt",
    "Kim-unwa FT2 (Vocal Hybrid)": "kimmel_unwa_ft2.ckpt",
    "BS-RoFormer ViperX (Vocals)": "model_bs_roformer_ep_368_sdr_12.9779.ckpt",

    # --- Multi-Stem ---
    "HTDemucs v4 FT (4 Stems)": "htdemucs_ft",
    "HTDemucs v4 6-Stems (Guitar/Piano)": "htdemucs_6s"
}

selected_model = models_map[model_choice]


# ------------------------------------------------------------
# Input Handling
# Priority: Spotify > YouTube > Google Drive folder
# ------------------------------------------------------------

files_to_process = []


if spotify_url.strip():

    print("🟢 Downloading track(s) from Spotify via SpotDL...")

    temp_spot_dir = "/content/spotdl_temp"

    if os.path.exists(temp_spot_dir):
        shutil.rmtree(temp_spot_dir)

    os.makedirs(temp_spot_dir, exist_ok=True)

    !spotdl download "{spotify_url}" --output "{temp_spot_dir}" --format wav

    files_to_process = glob.glob(f"{temp_spot_dir}/*.wav")

    if not files_to_process:
        raise FileNotFoundError(
            "❌ Unable to download track(s) from Spotify."
        )


elif youtube_url.strip():

    print("📥 Downloading audio from YouTube...")

    temp_yt_file = "/content/yt_temp.wav"

    if os.path.exists(temp_yt_file):
        os.remove(temp_yt_file)

    !yt-dlp -x --audio-format wav -o "{temp_yt_file}" "{youtube_url}"

    if os.path.exists(temp_yt_file):
        files_to_process = [temp_yt_file]
    else:
        raise FileNotFoundError(
            "❌ Unable to download audio from YouTube."
        )


elif audio_input_folder.strip():

    if not os.path.exists(audio_input_folder):
        raise FileNotFoundError(
            f"❌ Input folder not found at: {audio_input_folder}"
        )

    audio_extensions = (
        ".mp3",
        ".wav",
        ".flac",
        ".m4a",
        ".aac",
        ".ogg",
        ".opus",
        ".wma",
        ".aiff"
    )

    files_to_process = [
        os.path.join(audio_input_folder, f)
        for f in os.listdir(audio_input_folder)
        if f.lower().endswith(audio_extensions)
    ]

    files_to_process.sort()

    if not files_to_process:
        raise FileNotFoundError(
            f"❌ No valid audio files found in: {audio_input_folder}"
        )

else:

    raise ValueError(
        "❌ No audio source selected. "
        "Please provide a Spotify URL, YouTube URL, or Input Audio folder."
    )


# ------------------------------------------------------------
# Configuration Summary
# ------------------------------------------------------------

print(f"\n📂 Found {len(files_to_process)} track(s) to process.")

print(f"🚀 Initializing model [{model_choice}]...")

print(
    f"⚙️ Config: "
    f"Overlap={overlap} | "
    f"Format={output_format.upper()} | "
    f"Chunk={chunk_size} | "
    f"TTA={use_tta} | "
    f"Full Stems={extract_instrumental}\n"
)


# ------------------------------------------------------------
# Initialize Separator
# ------------------------------------------------------------

separator = Separator(
    output_dir=output_folder,
    output_format=output_format.upper(),
    output_single_stem=None if extract_instrumental else "Vocals"
)


# ------------------------------------------------------------
# Apply Processing Parameters
# ------------------------------------------------------------

separator.roformer_overlap = overlap
separator.mdx_overlap = overlap
separator.mdxc_overlap = overlap

separator.roformer_segment_size = chunk_size
separator.mdx_segment_size = chunk_size


if use_tta:
    separator.mdx_enable_tta = True
    separator.vr_enable_tta = True


# ------------------------------------------------------------
# Load AI Model
# ------------------------------------------------------------

try:

    separator.load_model(selected_model)

except Exception as e:

    print(
        f"⚠️ Primary model loading failed: {e}\n"
        "🔄 Attempting fallback model name..."
    )

    short_model_name = selected_model.replace(
        "melband_roformer_",
        ""
    )

    separator.load_model(short_model_name)


# ------------------------------------------------------------
# Process All Files
# ------------------------------------------------------------

for idx, file_path in enumerate(files_to_process, 1):

    track_name = os.path.splitext(
        os.path.basename(file_path)
    )[0]

    track_output_dir = os.path.join(
        output_folder,
        track_name
    )

    os.makedirs(
        track_output_dir,
        exist_ok=True
    )

    separator.output_dir = track_output_dir

    print(
        f"[{idx}/{len(files_to_process)}] 🎵 "
        f"Processing: {track_name}..."
    )

    try:

        output_files = separator.separate(file_path)

        print(
            f"   ✅ Saved stems to: "
            f"`{track_output_dir}`\n"
        )

    except Exception as e:

        print(
            f"   ❌ Failed to process {track_name}: {e}\n"
        )


# ------------------------------------------------------------
# Finished
# ------------------------------------------------------------

print("🔥 ALL SEPARATIONS COMPLETED!")

print(
    f"📁 Check your main output directory on Google Drive: "
    f"`{output_folder}`"
)

# 📖 User Guide & Parameter Breakdown

---

### ⚙️ What Do the First Two Cells Do?

#### **Step 1: Setup & Environment Prep**
- **Mounts Google Drive**: Connects your Drive to `/content/drive` so the script can access your audio files and save stems directly to the cloud.
- **Installs Tooling**: Auto-installs `audio-separator` (the SOTA GPU-supported AI backend), `yt-dlp` (for YouTube ripping), and `spotdl` (for Spotify downloading).
- **Auto-creates Folders**: Generates `Input_Audio` (where you drop tracks) and `Stems_Output` (where processed stems land) directly on your Drive.

#### **Step 2: Configuration & Stem Separator Execution**
- **Audio Ingestion**: Prioritizes Spotify URL > YouTube URL > Batch processing all files inside your `Input_Audio` Drive folder.
- **Model Execution**: Loads the AI model weights into GPU memory (T4) **once** and sequentially splits tracks one after another (saving tons of render time).
- **Clean File Organization**: Automatically creates a dedicated subfolder for each track inside `Stems_Output`, keeping your Drive clean and organized.

---

### 🎛️ Detailed Parameter Guide (Step 2)

#### **1. 📁 Audio Sources**
You have three input options with a strict hierarchy:
- **Spotify URL**: Highest priority. Paste a track, album, or playlist link. Uses `spotdl` to match and download high-quality audio.
- **YouTube URL**: Second priority. Paste any YouTube link to download and process audio on the fly in WAV format.
- **Input Folder Path**: If Spotify and YouTube inputs are blank, the script scans this Drive folder and batch-processes **all** valid audio files (`.mp3`, `.wav`, `.flac`, `.m4a`, etc.) in a single run.

---

#### **2. 🤖 AI Model Selection**

- **MelBand-RoFormer BigBeta Series (by pcunwa)**:
  - **BigBeta 7**: Absolute SOTA for vocals. Ultra-clean bleed removal without flattening high-frequency air or vocal transients.
  - **BigBeta 6X**: Surgical isolation. Zero-bleed focus, perfect for isolated acapellas in mashups/bootlegs where the vocal plays solo.
  - **BigBeta 6**: The ideal sweet spot between vocal fullness and clean instrumental rejection.
  - **BigBeta 5e**: Focused on vocal warmth (`Enhanced Fullness`). Keeps the low-end warmth of the lead vocal, but might leave minor instrumental bleed if heavy synths are present.
  - **BigBeta 4 / 3**: Classic fallback models if newer versions produce weird artifacts on specific tracks.

- **BS-RoFormer Series (Leap & ViperX)**:
  - Alternative SOTA architectures. **Leap** boasts insane SDR (signal-to-distortion ratio) scores and can outperform RoFormer on complex electronic tracks with heavy synth layers.

- **HTDemucs v4 (4-Stems / 6-Stems)**:
  - Use this when you need full multitrack separation (Drums, Bass, Guitar, Piano, Other) instead of just Vocal/Instrumental split.

---

#### **3. ⚙️ Advanced Parameters**

- **Overlap (1 - 40)**:
  - Controls how many times the AI overlaps analysis windows at segment boundaries to prevent clicks or seam artifacts.
  - *Pro Tip*: Stick to **`4` to `8`**. Going over `10` exponentially increases render times with almost zero noticeable gain.

- **Output Format (WAV / FLAC / MP3)**:
  - **WAV**: Uncompressed 24/32-bit audio. Ideal for dropping straight into your DAW (FL Studio, REAPER, etc.).
  - **FLAC**: Lossless compression (same audio quality as WAV, ~50% smaller file size).
  - **MP3**: Lossy compression, only use if you're running tight on Drive storage.

- **Chunk Size (Segment Size)**:
  - Processing window size (`112455`, `352800`, `485100`, `529200`, `661500`) loaded into GPU VRAM.
  - *Pro Tip*: Standard **`352800`** or **`485100`** works great. If Colab crashes with a `CUDA Out of Memory` error (especially with heavy models or ultra-long tracks), lower it to **`112455`**.

- **Use TTA (Test-Time Augmentation)**:
  - Runs a second pass with phase inversion to catch hidden frequencies and bleed.
  - *Pros*: Slightly cleaner acapellas.
  - *Cons*: Exactly doubles the processing time per track.

- **Extract Instrumental**:
  - **Enabled (True)**: Saves both `Vocals` and `Instrumental` stems.
  - **Disabled (False)**: Outputs only the isolated vocal stem, saving processing time and storage space.

---

*✨ Project built with the help of AI.*